# Campaign Effectiveness Analysis - Regression Extension

Extends the campaign effectiveness analysis using linear regression to 
control for baseline spend across the full dataset, rather than a manually 
matched subset.

## Question
Does Campaign 18 predict a real increase in household spend, once baseline 
spend is statistically controlled for — using the full sample rather than a 
narrowed, matched band?

## Method

Rebuilt the campaign and control group spend data (before vs. during 
Campaign 18) from the original analysis. Combined both groups into a single 
dataset with a `campaign_exposure` flag (1 = campaign, 0 = control), and 
fitted an OLS linear regression:

`change in spend = baseline spend + campaign exposure`

This allows the effect of campaign exposure to be estimated while 
mathematically holding baseline spend constant — using all 2,101 households, 
rather than the £100–£300 matched band used in the earlier analysis.

In [1]:
import pandas as pd
import duckdb
from scipy import stats
import statsmodels.api as sm

In [2]:
!pip install statsmodels

In [3]:
query_campaign = """
SELECT
    household_key,
    CASE 
        WHEN DAY >= 532 AND DAY < 587 THEN 'before'
        WHEN DAY >= 587 AND DAY < 642 THEN 'during'
    END AS period,
    SUM(SALES_VALUE) AS total_spend    
FROM 'transaction_data.csv' 
WHERE household_key IN (
    SELECT household_key FROM 'campaign_table.csv' WHERE CAMPAIGN = 18  
)
AND DAY >= 532 AND DAY < 642
GROUP BY household_key, period
"""
result_campaign = duckdb.sql(query_campaign).df()

query_control = """
SELECT
    household_key,
    CASE 
        WHEN DAY >= 532 AND DAY < 587 THEN 'before'
        WHEN DAY >= 587 AND DAY < 642 THEN 'during'
    END AS period,
    SUM(SALES_VALUE) AS total_spend
FROM 'transaction_data.csv'
WHERE household_key NOT IN (
    SELECT household_key FROM 'campaign_table.csv' 
    WHERE CAMPAIGN IN (13,14,15,16,17,18,19,20,21,22)
)
AND DAY >= 532 AND DAY < 642
GROUP BY household_key, period
"""
result_control = duckdb.sql(query_control).df()

pivot = result_campaign.pivot(index='household_key', columns='period', values='total_spend').fillna(0)
control_pivot = result_control.pivot(index='household_key', columns='period', values='total_spend').fillna(0)

pivot['change'] = pivot['during'] - pivot['before']
control_pivot['change'] = control_pivot['during'] - control_pivot['before']

print(pivot.shape)
print(control_pivot.shape)

(1123, 3)
(978, 3)


In [4]:
pivot['campaign_exposure'] = 1
control_pivot['campaign_exposure'] = 0

combined = pd.concat([pivot, control_pivot])
print(combined.shape)
print(combined.head())

(2101, 4)
period         before  during  change  campaign_exposure
household_key                                           
1              342.71  465.67  122.96                  1
2              236.35  201.86  -34.49                  1
6              465.11  406.07  -59.04                  1
7              392.31  299.41  -92.90                  1
8              464.81  478.20   13.39                  1


In [5]:
X = combined[['before', 'campaign_exposure']]
X = sm.add_constant(X)  # adds an intercept term, required for proper regression
y = combined['change']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 change   R-squared:                       0.019
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     20.62
Date:                Thu, 16 Jul 2026   Prob (F-statistic):           1.36e-09
Time:                        00:39:17   Log-Likelihood:                -13854.
No. Observations:                2101   AIC:                         2.771e+04
Df Residuals:                    2098   BIC:                         2.773e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                31.4804      5.87

## Interpretation

Holding baseline spend constant, campaign exposure is associated with a 
statistically significant £38.77 increase in spend. This aligns directionally 
with the matched-band analysis (+£51.79 for the £100-£300 baseline group), 
using the full dataset rather than a narrowed subset - two different methods 
converging on the same conclusion.

The baseline spend coefficient (-£0.08 per £1) confirms a regression-to-the-mean 
pattern: households that already spent more tended to show smaller increases, 
independent of campaign exposure. This is the same confound the matched-band 
approach was designed to control for, and the regression result validates that 
it was a real, worthwhile correction.

## Limitations

- R-squared is low (0.019), meaning baseline spend and campaign exposure 
  together explain only a small share of the variation in spend change. Other 
  factors (product mix, life circumstances, other promotions) likely drive 
  most of the variation and are not captured here.
- Statsmodels flagged a high condition number, suggesting possible 
  multicollinearity or scaling issues between variables — a fuller analysis 
  would standardise variables and check this more rigorously.
- Regression assumes a straight-line relationship between baseline spend and 
  change in spend across the full range; this may not hold for very low or 
  very high spenders.

## Conclusion

Two independent methods - a baseline-matched control group and a linear 
regression across the full sample - both indicate that Campaign 18 was 
associated with a genuine, statistically significant increase in household 
spend, after accounting for pre-existing differences in baseline spend. This 
convergence strengthens confidence in the finding beyond what either method 
would support alone, while the limitations above outline where further work 
(e.g. more robust regression diagnostics, non-linear modelling) could extend 
the analysis further.

## Tools
Python (pandas, statsmodels), SQL (DuckDB), Jupyter Notebook